In [ ]:
import os
import math
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch_geometric.utils import from_networkx

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from xgboost import XGBClassifier

from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, matthews_corrcoef

In [ ]:
data = pd.read_csv('../out_files/alphafold/featurevector_alphafold.txt', sep='\t', low_memory=False)
data

In [ ]:
data.replace('[]', np.nan, inplace=True)
data.replace('hit', np.nan, inplace=True)

In [ ]:
nan_counts = data.isna().sum()

plt.figure(figsize=(20, 15))
nan_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('NaN for Column', fontsize=14)
plt.xlabel('Columns', fontsize=12)
plt.ylabel('Counts', fontsize=12)
plt.xticks(rotation=90, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
missing_percent = data.isna().mean() * 100
columns_to_remove = missing_percent[missing_percent > 20].index
print(columns_to_remove)

In [ ]:
data['identifier'] = data['prot_uniprotAcc'] + '_' + data['wt_residue'] + '_' + data['position'].astype(str) + '_' + data['mut_residue']
data.set_index('identifier', inplace=True)

In [ ]:
data_cleaned = data.drop(columns=columns_to_remove)
data_cleaned = data_cleaned.drop(columns=['domains_all', 'meta_merged', 'prot_uniprotAcc', 'position', 'mut_residue', 'wt_residue'])
data_cleaned.iloc[:, -4:] = data_cleaned.iloc[:, -4:].astype(float)

In [ ]:
data_cleaned.info()

In [ ]:
#data_cleaned.to_csv('dataset_cleaned.csv')

## Train & Test

In [ ]:
dataTrain = pd.read_csv('../dataset/data_train.csv', index_col='identifier')
dataTrain

In [ ]:
data_merged = pd.merge(data_cleaned, dataTrain['function'], left_index=True, right_index=True, how='inner')
data_merged.to_csv('additional_features.csv')

In [ ]:
# from tqdm import tqdm
# features = []

# for index in tqdm(data_merged.index, desc='Preparing features'):
#     if index + '.pt' in os.listdir('../embedding/graphs'):
#         graph_path = os.path.join('../embedding/graphs', f"{index}.pt")
#         graph = torch.load(graph_path)[0]

#         tmp = from_networkx(graph)

#         features.append(tmp.features)

In [ ]:
for cv, split in enumerate(os.listdir('../splits')):
    if split.endswith('.csv'):
        print(f'Cross Validation: Fold {cv}')
        split_data = pd.read_csv(f'{os.path.join("../splits", split)}')
        
        ids_train = list(split_data['train'])
        ids_val = [x for x in list(split_data['val']) if not (isinstance(x, float) and math.isnan(x))]

        filtered_ids_train = [id_ for id_ in ids_train if id_ in data_merged.index]
        filtered_ids_val = [id_ for id_ in ids_val if id_ in data_merged.index]

        outcome_train = data_merged.loc[filtered_ids_train]['function'].tolist()
        outcome_val = data_merged.loc[filtered_ids_val]['function'].tolist()

        outcome_train = [outcome for outcome in outcome_train if outcome != -999]
        outcome_val = [outcome for outcome in outcome_val if outcome != -999]

        ids_train_filtered = [filtered_ids_train[i] for i in range(len(outcome_train)) if outcome_train[i] != -999]
        ids_val_filtered = [filtered_ids_val[i] for i in range(len(outcome_val)) if outcome_val[i] != -999]

        X_train = data_merged.loc[ids_train_filtered].drop('function', axis=1)
        X_val = data_merged.loc[ids_val_filtered].drop('function', axis=1)

        imputer = SimpleImputer(strategy='mean')
        X_train_imputed = imputer.fit_transform(X_train)
        X_val_imputed = imputer.transform(X_val)

        #model = RandomForestClassifier(n_estimators=100, random_state=42)
        model = XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=42
        )
        
        model.fit(X_train_imputed, outcome_train)

        y_pred = model.predict(X_val_imputed)

        mcc = matthews_corrcoef(outcome_val, y_pred)
        print(f'MCC: {mcc}')

In [ ]:
param_grid = {
    'n_estimators': [10, 20, 30, 100, 200, 300],
    'learning_rate': [0.1, 0.2, 0.01],
    'max_depth': [2,3,4,5]
}

mcc_scorer = make_scorer(matthews_corrcoef)

for cv, split in enumerate(os.listdir('../splits')):
    if split.endswith('.csv'):
        print(f'Cross Validation: Fold {cv}')
        split_data = pd.read_csv(f'{os.path.join("../splits", split)}')
        
        ids_train = list(split_data['train'])
        ids_val = [x for x in list(split_data['val']) if not (isinstance(x, float) and math.isnan(x))]

        filtered_ids_train = [id_ for id_ in ids_train if id_ in data_merged.index]
        filtered_ids_val = [id_ for id_ in ids_val if id_ in data_merged.index]

        outcome_train = data_merged.loc[filtered_ids_train]['function'].tolist()
        outcome_val = data_merged.loc[filtered_ids_val]['function'].tolist()

        outcome_train = [outcome for outcome in outcome_train if outcome != -999]
        outcome_val = [outcome for outcome in outcome_val if outcome != -999]

        ids_train_filtered = [filtered_ids_train[i] for i in range(len(outcome_train)) if outcome_train[i] != -999]
        ids_val_filtered = [filtered_ids_val[i] for i in range(len(outcome_val)) if outcome_val[i] != -999]

        X_train = data_merged.loc[ids_train_filtered].drop('function', axis=1)
        X_val = data_merged.loc[ids_val_filtered].drop('function', axis=1)

        imputer = SimpleImputer(strategy='mean')
        X_train_imputed = imputer.fit_transform(X_train)
        X_val_imputed = imputer.transform(X_val)

        pos_class_weight = (len(outcome_train) - np.sum(outcome_train)) / np.sum(outcome_train)

        rf = XGBClassifier(objective='binary:logistic',
                           scale_pos_weight=pos_class_weight,
                           random_state=42)

        grid_search = GridSearchCV(
            estimator=rf,
            param_grid=param_grid,
            scoring=mcc_scorer,
            n_jobs=-1
        )

        grid_search.fit(X_train_imputed, outcome_train)

        best_model = grid_search.best_estimator_
        print(f'Best params for Fold {cv}: {grid_search.best_params_}')

        #y_proba = model.predict_proba(X_val_imputed)[:, 1]

        #threshold = 0.6
        #y_pred = (y_proba >= threshold).astype(int)

        y_pred = best_model.predict(X_val_imputed)

        mcc = matthews_corrcoef(outcome_val, y_pred)
        print(f'MCC: {mcc}\n')